In [ ]:
import os
import glob
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

In [ ]:
# --- CONFIGURATION DES PATHS ---
DATA_DIR = "data/fastq"  # Dossier contenant vos .fastq.gz
RESULT_DIR = "results_profiling"
os.makedirs(RESULT_DIR, exist_ok=True)

In [ ]:
# Chemins des bases de données (à adapter selon votre cluster)
SYLPH_DB = "/shared/bank/gtdb/current/sylph/gtdb-r214.sylphdb"
SINGLEM_DB = "/shared/bank/singlem/current/metapackage" # Si nécessaire

In [ ]:
# Liste des échantillons (on suppose un format sample_R1.fastq.gz)
fastq_r1s = sorted(glob.glob(f"{DATA_DIR}/*_R1.fastq.gz"))
samples = [os.path.basename(f).split('_R1')[0] for f in fastq_r1s]

In [ ]:
# Dossier spécifique pour Sylph
sylph_out = f"{RESULT_DIR}/sylph"
os.makedirs(sylph_out, exist_ok=True)

In [ ]:
for r1 in fastq_r1s:
    sample = os.path.basename(r1).split('_R1')[0]
    r2 = r1.replace("_R1", "_R2")
    output = f"{sylph_out}/{sample}.tsv"
    
    print(f"🚀 Processing {sample} with Sylph...")
    
    # On charge le module et on lance l'outil dans la même ligne
    !module load sylph && sylph profile {SYLPH_DB} {r1} {r2} -t 8 -o {output}

In [ ]:
# Agrégation des résultats Sylph
sylph_tables = []
for s in samples:
    path = f"{sylph_out}/{s}.tsv"
    if os.path.exists(path):
        df = pd.read_csv(path, sep='\t')
        # On garde le nom du taxon et la proportion (abondance relative)
        df = df[['tax_name', 'proportion']].rename(columns={'proportion': s})
        sylph_tables.append(df.set_index('tax_name'))

In [ ]:
df_sylph_final = pd.concat(sylph_tables, axis=1).fillna(0)
df_sylph_final.to_csv(f"{RESULT_DIR}/combined_sylph_abundance.tsv", sep='\t')

In [ ]:
# 📊 3. Analyse Comparative & Clustering

In [ ]:
# Choix de la table à analyser
data_to_plot = df_sylph_final.copy()

# Filtrage : On ne garde que les taxons qui atteignent au moins 1% quelque part
data_filtered = data_to_plot[data_to_plot.max(axis=1) >= 0.01]

print(f"Nombre de taxons après filtrage (>1%) : {len(data_filtered)}")

In [ ]:
# --- CLUSTERING HIERARCHIQUE ---
# Calcul de la distance (Euclidienne) et du lien (Ward)
Z = linkage(pdist(data_filtered.T), method='ward')

# Affichage du Clustermap
g = sns.clustermap(data_filtered, 
                   method='ward', 
                   cmap="YlGnBu", 
                   figsize=(12, 10),
                   xticklabels=True, 
                   yticklabels=True,
                   cbar_kws={'label': 'Abondance Relative'})

plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right')
plt.suptitle(f"Heatmap de clustering hiérarchique (Top {len(data_filtered)} taxons)", fontsize=16)
plt.show()